In [ ]:
# BLOCK 0 - Dependencies
#pip install dicom2nifti pydicom

In [ ]:
# BLOCK 1 – Convert DICOMs to NIfTIs while preserving orientation

import os
import shutil
import glob
import pydicom
import dicom2nifti
import dicom2nifti.settings as settings

# EDIT PATHS HERE
DICOM_ROOT = r"PATH_TO_DICOMS"
NIFTI_ROOT = r"OUTPUT_NIFTI_PATH"

settings.reorient_nifti = True


def _match_series(series_dir):
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    if len(files) <= 50:
        return False
    try:
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
    except Exception:
        return False
    image_type = ds.get("ImageType", [])
    if isinstance(image_type, str):
        image_type = [v.strip() for v in image_type.split("\\")]
    tags = {v.upper() for v in image_type}
    return {"ORIGINAL", "PRIMARY", "AXIAL"}.issubset(tags)


def pick_series(patient_dir):
    candidates = []
    if glob.glob(os.path.join(patient_dir, "*.dcm")):
        candidates.append(patient_dir)
    for name in os.listdir(patient_dir):
        path = os.path.join(patient_dir, name)
        if os.path.isdir(path):
            candidates.append(path)
    for sdir in candidates:
        if _match_series(sdir):
            return sdir
    return None


for pid in os.listdir(DICOM_ROOT):
    src_dir = os.path.join(DICOM_ROOT, pid)
    if not os.path.isdir(src_dir):
        continue

    series_dir = pick_series(src_dir)
    if not series_dir:
        print(f"No suitable series found for {pid}")
        continue

    tmp_dir = os.path.join(NIFTI_ROOT, pid)

    if os.path.isdir(tmp_dir):
        print(f"Deleting old temp output: {tmp_dir}")
        shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir, exist_ok=True)

    print(f"\nConverting {pid}: {series_dir} -> {tmp_dir}")
    try:
        dicom2nifti.convert_directory(series_dir, tmp_dir, compression=True)
    except Exception as e:
        print(f"Conversion failed for {pid}: {e}")
        shutil.rmtree(tmp_dir, ignore_errors=True)
        continue

    nii_files = glob.glob(os.path.join(tmp_dir, "*.nii*"))
    if len(nii_files) == 1:
        src_nii = nii_files[0]
        dst_nii = os.path.join(NIFTI_ROOT, f"{pid}.nii.gz")
        os.replace(src_nii, dst_nii)
        print(f"  Saved as {dst_nii}")
    else:
        print(f"Unexpected files in {tmp_dir}: {nii_files}")

    shutil.rmtree(tmp_dir, ignore_errors=True)

print("\nAll DICOMs converted to NIfTIs.")
